# Appendix 4. Matplotlib Figure와 Axes 기초

## 1. Goal

pandas의 `.plot()` 결과를 다루려면 먼저 Matplotlib의 객체 구조를 알아야 합니다. 이 notebook은 chart 종류를 많이 소개하기보다 `Figure`, `Axes`, `Axis`, `Artist`의 관계와 figure 설정, `subplot` 구성을 집중적으로 연습합니다.

학습이 끝나면 pandas가 반환한 `Axes`에 제목·축·범위·grid·legend를 추가하고, 그 Axes가 속한 Figure를 설정하거나 저장할 수 있습니다.

## 2. Setup

프로젝트 lock의 Matplotlib 버전은 3.11.0입니다.

모든 데이터는 고정된 Python list입니다. figure를 파일로 저장하는 예제도 메모리 buffer를 사용하므로 workspace에 생성 파일을 남기지 않습니다.

In [ ]:
from io import BytesIO

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.artist import Artist
from matplotlib.axes import Axes
from matplotlib.figure import Figure

matplotlib.__version__


## 3. Steps

각 `###` 절은 Matplotlib의 역할별 개념군을, `####` 절은 하나의 실행 예제로 확인할 세부 학습 단위를 나타냅니다. 객체 모델을 이해한 뒤 Figure, subplot, Axes, 출력 설정 순서로 진행합니다.

### 3-1. Matplotlib 객체 모델

#### 3-1-1. Figure, Axes, Axis, Artist 관계

`Figure`는 최상위 canvas 역할을 하는 컨테이너입니다. Figure 안에는 하나 이상의 `Axes`가 있고, Axes는 실제 데이터가 그려지는 영역입니다. Axes 안의 `xaxis`, `yaxis`는 tick과 tick label을 관리하는 `Axis` 객체입니다. 선, 막대, 글자, legend 같은 화면 요소는 모두 `Artist` 계열 객체입니다.

이름이 비슷하지만 `Axes`는 chart 영역 전체이고, `Axis`는 x축 또는 y축 하나라는 차이를 기억하세요.

In [ ]:
object_figure = plt.figure(figsize=(6, 3), dpi=100, layout="constrained")
object_axes = object_figure.add_subplot(1, 1, 1)
line_artist, = object_axes.plot([0, 1, 2, 3], [78, 82, 86, 84], marker="o", label="HR")
object_axes.legend()
print(
    {
        "figure_type": type(object_figure).__name__,
        "axes_type": type(object_axes).__name__,
        "xaxis_type": type(object_axes.xaxis).__name__,
        "line_artist_type": type(line_artist).__name__,
        "axes_in_figure": len(object_figure.axes),
    }
)


### 3-2. Figure 생성과 설정

#### 3-2-1. 크기·해상도·배경·전체 layout 설정

`plt.figure()`는 Figure를 만들고 현재 pyplot 상태에 연결하는 편의 API입니다. `figsize=(width, height)`는 inch 단위이고, raster 출력의 대략적인 pixel 크기는 inch와 `dpi`의 곱입니다. `facecolor`는 Figure 배경, `layout="constrained"`는 title·tick label·legend가 서로 겹치지 않도록 Axes 배치를 조정합니다.

Figure를 만든 뒤 `add_subplot()`으로 Axes를 추가할 수 있습니다. `suptitle`, `supxlabel`, `supylabel`은 개별 Axes가 아니라 Figure 전체에 적용됩니다.

In [ ]:
configured_figure = plt.figure(
    figsize=(7, 3.5),
    dpi=120,
    facecolor="white",
    layout="constrained",
)
configured_axes = configured_figure.add_subplot(1, 1, 1)
configured_axes.plot([0, 30, 60, 90], [78, 81, 86, 84], marker="o")
configured_figure.suptitle("ICU measurement overview")
configured_figure.supxlabel("Minute since record start")
configured_figure.supylabel("Heart rate [BPM]")
print(
    {
        "size_inches": configured_figure.get_size_inches().tolist(),
        "dpi": configured_figure.dpi,
        "approx_pixels": (
            configured_figure.get_size_inches() * configured_figure.dpi
        ).astype(int).tolist(),
        "layout_engine": type(configured_figure.get_layout_engine()).__name__,
    }
)


### 3-3. Subplot 구성

#### 3-3-1. plt.subplots로 여러 Axes 만들기

`plt.subplots()`는 Figure와 Axes grid를 함께 반환합니다. `nrows`, `ncols`가 1보다 크면 `axes`는 보통 배열이므로 `axes[row, column]` 또는 `axes.flat`으로 접근합니다. `sharex`와 `sharey`는 같은 의미·단위를 가진 축을 공유할 때 사용합니다.

`gridspec_kw`의 `width_ratios`, `height_ratios`는 subplot의 상대 크기를 조절합니다. 수치는 pixel이 아니라 서로의 비율입니다.

In [ ]:
subplot_figure, subplot_axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(9, 5),
    dpi=100,
    sharex="col",
    layout="constrained",
    gridspec_kw={"width_ratios": [2, 1], "height_ratios": [1, 1]},
)
subplot_axes[0, 0].plot([0, 30, 60, 90], [78, 81, 86, 84], marker="o")
subplot_axes[0, 1].bar(["P101", "P102", "P103"], [4, 3, 5])
subplot_axes[1, 0].hist([72, 74, 78, 81, 84, 86, 88, 91, 96], bins=4)
subplot_axes[1, 1].scatter([1.2, 1.8, 2.5, 3.1], [78, 82, 88, 92])
subplot_figure.suptitle("Multiple Axes created by plt.subplots")
print({"axes_shape": subplot_axes.shape, "axes_count": len(subplot_figure.axes)})


### 3-4. Axes 설정과 Artist 수정

#### 3-4-1. title, label, limit, grid, tick, legend

Axes가 있으면 plotting 함수의 출처가 Matplotlib이든 pandas이든 같은 방식으로 후처리할 수 있습니다. `set()`은 여러 속성을 한 번에 지정하고, `set_xlim`·`set_ylim`은 표시 범위를, `grid`는 보조선을, `tick_params`는 tick과 label 모양을 조절합니다.

범위를 수동 설정할 때는 실제 데이터를 잘라내지 않는지 먼저 확인하세요. legend는 `label`이 있는 Artist를 설명하므로, `ax.legend()`만 호출하기 전에 plot에 의미 있는 label을 지정해야 합니다.

In [ ]:
axes_figure, axes_configured = plt.subplots(
    figsize=(7, 3.5), dpi=100, layout="constrained"
)
configured_line, = axes_configured.plot(
    [0, 30, 60, 90],
    [78, 81, 86, 84],
    marker="o",
    label="Heart rate",
)
axes_configured.set(
    title="Heart rate over observation time",
    xlabel="Minute since record start",
    ylabel="Heart rate [BPM]",
)
axes_configured.set_xlim(0, 90)
axes_configured.set_ylim(70, 100)
axes_configured.grid(axis="y", alpha=0.3)
axes_configured.tick_params(axis="x", labelrotation=0)
axes_configured.legend(loc="upper left")


#### 3-4-2. Axes 메서드가 반환하는 Artist 다루기

Axes의 plotting 메서드는 화면에 그린 Artist를 반환합니다. `plot`은 `Line2D`, `bar`는 막대 Rectangle을 담은 container를 반환합니다. 반환값을 보관하면 나중에 특정 선의 두께나 특정 막대의 색만 바꿀 수 있습니다. pandas plotting이 반환하는 것은 보통 Artist가 아니라 그 Artist들을 담고 있는 `Axes`라는 차이도 다음 notebook에서 확인합니다.

In [ ]:
artist_figure, artist_axes = plt.subplots(figsize=(7, 3.5), layout="constrained")
artist_line, = artist_axes.plot([0, 1, 2], [78, 86, 84], label="HR")
bar_container = artist_axes.bar([0, 1, 2], [2, 1, 3], alpha=0.25, label="Count")
artist_line.set_linewidth(2.5)
bar_container.patches[0].set_facecolor("tab:orange")
artist_axes.legend()
print(
    {
        "line_type": type(artist_line).__name__,
        "bar_count": len(bar_container.patches),
        "axes_lines": len(artist_axes.lines),
        "axes_patches": len(artist_axes.patches),
    }
)


### 3-5. Figure layout과 출력

#### 3-5-1. Layout 선택과 subplot 간격

여러 Axes에서는 title, tick label, legend가 겹치기 쉽습니다. 공식 문서는 일반적인 figure에 `layout="constrained"` 사용을 안내합니다. layout engine은 Axes를 만든 뒤가 아니라 Figure 또는 subplots를 만들 때 활성화하는 것이 안전합니다.

constrained layout을 사용한 뒤 `plt.tight_layout()`을 다시 호출하면 constrained layout이 꺼지므로 두 방식을 섞지 않습니다. 간격을 세밀하게 조절해야 하면 `fig.get_layout_engine().set(...)`을 사용합니다.

In [ ]:
layout_figure, layout_axes = plt.subplots(
    1, 2, figsize=(8, 3), layout="constrained"
)
layout_axes[0].plot([0, 1, 2], [1, 3, 2])
layout_axes[1].bar(["A", "B", "C"], [3, 1, 2])
layout_axes[0].set_title("Trend")
layout_axes[1].set_title("Counts")
layout_figure.get_layout_engine().set(w_pad=4 / 72, h_pad=4 / 72)
type(layout_figure.get_layout_engine()).__name__


#### 3-5-2. Figure 저장 설정

`Figure.savefig()`는 Figure 전체를 PNG, SVG, PDF 같은 형식으로 저장합니다. raster 형식의 해상도는 `dpi`, 바깥 여백은 `bbox_inches`와 `pad_inches`, 배경 투명도는 `transparent`로 조절합니다. 이 예제는 `BytesIO`에 PNG를 써서 실제 파일 없이 저장 API를 검증합니다.

In [ ]:
image_buffer = BytesIO()
axes_figure.savefig(
    image_buffer,
    format="png",
    dpi=150,
    bbox_inches="tight",
    pad_inches=0.1,
    transparent=False,
)
saved_png_bytes = image_buffer.tell()
saved_png_bytes


## 4. Checks

객체 관계, subplot 개수, Figure 설정과 메모리 저장 결과를 확인합니다. 시각화 테스트에서는 pixel을 비교하기보다 생성된 객체와 핵심 속성을 검증하는 편이 안정적입니다.

In [ ]:
assert isinstance(object_figure, Figure)
assert isinstance(object_axes, Axes)
assert isinstance(line_artist, Artist)
assert object_axes.figure is object_figure
assert configured_figure.get_size_inches().tolist() == [7.0, 3.5]
assert configured_figure.dpi == 120
assert subplot_axes.shape == (2, 2)
assert len(subplot_figure.axes) == 4
assert configured_line in axes_configured.lines
assert len(bar_container.patches) == 3
assert saved_png_bytes > 0
print("All appendix checks passed.")
plt.close("all")


## 5. Next Steps

- Figure는 전체 크기·해상도·layout·저장을, Axes는 데이터 표현과 축 설정을 담당합니다.
- 여러 chart는 `plt.subplots()`로 Axes grid를 먼저 설계하고 각 Axes에 하나의 질문을 배치합니다.
- plotting 메서드의 반환 객체를 보관하면 특정 Artist를 나중에 수정하거나 테스트할 수 있습니다.
- 다음 notebook에서는 pandas `.plot()`이 반환한 Axes를 받아 여기서 배운 API로 후처리합니다.

## 6. References

이 notebook의 설명과 예제는 Matplotlib 공식 문서를 기준으로 작성했습니다.

- [Figures and backends](https://matplotlib.org/stable/users/explain/figure/index.html)
- [Introduction to Axes](https://matplotlib.org/stable/users/explain/axes/axes_intro.html)
- [Introduction to Artists](https://matplotlib.org/stable/users/explain/artists/artist_intro.html)
- [Constrained layout guide](https://matplotlib.org/stable/users/explain/axes/constrainedlayout_guide.html)
- [Figure.savefig API](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.savefig.html)